# Comprensión de los datos y análisis exploratorio (EDA)
Proyecto final de Machine Learning: predicción de pago a tiempo de créditos.
Variable objetivo: `Pago_atiempo` (1 = pagó a tiempo, 0 = no pagó a tiempo).

# PARTE 1. CONOCER LOS DATOS

## 1.1 Cargar los datos

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

df = pd.read_csv("../Base_de_datos.csv")
print(df.shape)
df.head()

(10763, 23)


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,2025-01-07 14:40:36,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,2025-01-09 11:18:48,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,2024-12-26 18:52:57,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,2024-12-04 14:20:50,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,2025-04-30 18:41:57,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


## 1.2 Resumen de cada variable

In [6]:
resumen = pd.DataFrame({
    "tipo_python": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "nulos": df.isna().sum(),
    "pct_nulos": (df.isna().mean() * 100).round(2)
})
resumen

,tipo_python,valores_unicos,nulos,pct_nulos
tipo_credito,int64,6,0,0.00
fecha_prestamo,str,10758,0,0.00
capital_prestado,float64,7306,0,0.00
plazo_meses,int64,18,0,0.00
edad_cliente,int64,54,0,0.00
tipo_laboral,str,2,0,0.00
salario_cliente,int64,1385,0,0.00
total_otros_prestamos,int64,1538,0,0.00
cuota_pactada,int64,9836,0,0.00
puntaje,float64,248,0,0.00


## 1.3 Clasificar las variables

### Caracterización de las variables

| Variable | Tipo de dato | Clasificación | Observación |
|---|---|---|---|
| tipo_credito | Categórica | Nominal, politómica (6 códigos) | Es un código de producto, no una cantidad |
| fecha_prestamo | Fecha y hora | Temporal | Se convierte a datetime |
| capital_prestado | Numérica | Continua | Monto en pesos |
| plazo_meses | Numérica | Discreta | Meses del crédito |
| edad_cliente | Numérica | Discreta | Años del cliente |
| tipo_laboral | Categórica | Nominal, dicotómica | Empleado / Independiente |
| salario_cliente | Numérica | Continua | Monto en pesos |
| total_otros_prestamos | Numérica | Continua | Monto en pesos |
| cuota_pactada | Numérica | Continua | Monto en pesos |
| puntaje | Numérica | Continua | Puntaje interno; se revisará posible fuga de datos |
| puntaje_datacredito | Numérica | Continua | 6 nulos |
| cant_creditosvigentes | Numérica | Discreta | Conteo de créditos |
| huella_consulta | Numérica | Discreta | Conteo de consultas |
| saldo_mora | Numérica | Continua | 156 nulos |
| saldo_total | Numérica | Continua | 156 nulos |
| saldo_principal | Numérica | Continua | 405 nulos |
| saldo_mora_codeudor | Numérica | Continua | Casi todo 0; 590 nulos |
| creditos_sectorFinanciero | Numérica | Discreta | Conteo de créditos |
| creditos_sectorCooperativo | Numérica | Discreta | Conteo de créditos |
| creditos_sectorReal | Numérica | Discreta | Conteo de créditos |
| promedio_ingresos_datacredito | Numérica | Continua | 27,2 % de nulos |
| tendencia_ingresos | Categórica | Ordinal, politómica (3) | Decreciente < Estable < Creciente; trae valores numéricos incorrectos; 27,2 % de nulos |
| Pago_atiempo | Numérica (0/1) | Dicotómica | **Variable objetivo** (1 pagó a tiempo, 0 no) |

**Conclusión:** la base tiene 1 variable de fecha (`fecha_prestamo`), 18 numéricas, 3 categóricas (`tipo_credito`, `tipo_laboral` y `tendencia_ingresos`) y 1 variable dicotómica `Pago_atiempo` que es la variable objetivo.

## 1.4 Revisar la columna tendencia_ingresos

In [7]:
df["tendencia_ingresos"].value_counts(dropna=False).head(10)

tendencia_ingresos
Creciente      5294
NaN            2932
Decreciente    1291
Estable        1188
0                 7
8315              6
1000000           4
9147              2
158042            1
3978              1
Name: count, dtype: int64

### Hallazgos en tendencia_ingresos

La variable debería tener solo tres categorías: Creciente, Estable y Decreciente.

- **Creciente:** 5.294 registros (49,2 %). Es la categoría más frecuente.
- **Decreciente:** 1.291 registros (12,0 %).
- **Estable:** 1.188 registros (11,0 %).
- **Valores vacíos (NaN):** 2.932 registros (27,2 %). Coincide con los vacíos de `promedio_ingresos_datacredito`, lo que sugiere que son clientes sin historial de ingresos en el buró.
- **Valores numéricos (0, 8315, 1000000, 9147, etc.):** 58 registros (0,5 %). No corresponden a ninguna categoría válida; parecen montos de dinero que se cargaron por error en esta columna (posible desplazamiento de datos al cargar).

**Decisión:** los 58 valores numéricos se tratan como nulos, porque no se puede saber a qué categoría pertenecían. Después de esto, la variable queda con 3 categorías y cerca de 27,8 % de nulos. Más adelante se decidirá si los nulos se imputan o se dejan como una categoría aparte ("Sin información").

## 1.5 Unificar los vacíos y corregir valores raros

In [8]:
valores_nulos = ["", " ", "NA", "N/A", "na", "null", "NULL", "None", "nan", "NaN", "-"]
df = df.replace(valores_nulos, np.nan)

validas = ["Creciente", "Estable", "Decreciente"]
mask = df["tendencia_ingresos"].notna() & ~df["tendencia_ingresos"].isin(validas)
print("Valores inválidos encontrados:", mask.sum())
df.loc[mask, "tendencia_ingresos"] = np.nan
df["tendencia_ingresos"].isna().mean().round(3)

Valores inválidos encontrados: 58


np.float64(0.278)

**Resultado:** se unificó la representación de nulos y se encontraron 58 valores inválidos en `tendencia_ingresos`, que se convirtieron en nulos. Ahora esta variable tiene 2.990 nulos (27,8 %).

## 1.6 Poner cada columna en su formato

In [9]:
df["fecha_prestamo"] = pd.to_datetime(df["fecha_prestamo"])
for c in ["tipo_credito", "tipo_laboral", "tendencia_ingresos"]:
    df[c] = df[c].astype("category")
df.dtypes

tipo_credito                           category
fecha_prestamo                   datetime64[us]
capital_prestado                        float64
plazo_meses                               int64
edad_cliente                              int64
tipo_laboral                           category
salario_cliente                           int64
total_otros_prestamos                     int64
cuota_pactada                             int64
puntaje                                 float64
puntaje_datacredito                     float64
cant_creditosvigentes                     int64
huella_consulta                           int64
saldo_mora                              float64
saldo_total                             float64
saldo_principal                         float64
saldo_mora_codeudor                     float64
creditos_sectorFinanciero                 int64
creditos_sectorCooperativo                int64
creditos_sectorReal                       int64
promedio_ingresos_datacredito           

## 1.7 Quitar variables que no aportan

In [10]:
df["saldo_mora_codeudor"].value_counts(dropna=False)

saldo_mora_codeudor
0.0       10170
NaN         590
30.0          1
470.0         1
2145.0        1
Name: count, dtype: int64

La variable `saldo_mora_codeudor` tiene solo 4 valores distintos. De los 10.173 registros con dato, 10.170 son 0, es decir, el 99,97 %. Además tiene 590 nulos (5,5 %). Al ser casi constante, no aporta información para distinguir a un cliente de otro y se elimina.

Nota: `saldo_mora` también es casi todo 0, pero no se elimina todavía: aunque tiene poca variación, se revisará su relación con `Pago_atiempo` en el análisis bivariable, porque los pocos casos con mora podrían ser justo los que no pagan.

In [11]:
df = df.drop(columns=["saldo_mora_codeudor"])

## 1.8 Revisión final: filas repetidas y vacíos que quedan

In [12]:
print("Filas duplicadas:", df.duplicated().sum())
nulos_final = df.isna().sum()
nulos_final[nulos_final > 0]

Filas duplicadas: 0


puntaje_datacredito                 6
saldo_mora                        156
saldo_total                       156
saldo_principal                   405
promedio_ingresos_datacredito    2930
tendencia_ingresos               2990
dtype: int64

**Resultado:** no hay filas duplicadas. Quedan nulos en 6 variables: `puntaje_datacredito` (6), `saldo_mora` y `saldo_total` (156 cada una), `saldo_principal` (405), `promedio_ingresos_datacredito` (2.930) y `tendencia_ingresos` (2.990). Estos nulos no se rellenan todavía: se decidirá en la etapa de ingeniería de características cómo tratarlos, porque los dos últimos superan el 27 %.